In [1]:
import datetime
from typing import Optional
from pydantic import BaseModel, Field
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.tracers import LangChainTracer

class TutorialSearchQuery(BaseModel):
    """Structured query for searching technical video tutorials."""

    query: str = Field(
        ...,
        description="Core semantic search query to compare against transcript embeddings."
    )
    published_year: Optional[int] = Field(
        None,
        description="Filter by publication year if explicitly specified."
    )
    min_views: Optional[int] = Field(
        None,
        description="Minimum view count threshold, inclusive."
    )
    language: Optional[str] = Field(
        None,
        description="Programming language constraint (e.g., 'python', 'javascript')."
    )

llm = ChatGroq(model="openai/gpt-oss-20b", temperature=0)
structured_llm = llm.with_structured_output(TutorialSearchQuery)

system_prompt = """You are an expert query constructor. Convert user questions
into structured search parameters for a technical video database.
Do not infer filters unless explicitly requested or clearly implied."""

prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{question}"),
])

query_constructor = prompt | structured_llm

tracer = LangChainTracer()

result = query_constructor.invoke({
    "question": "Find Python videos on RAG from scratch published in 2024 with at least 5000 views"
}, config={"callbacks": [tracer]})

print(result)

query='RAG from scratch' published_year=2024 min_views=5000 language='Python'
